In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Technologies, Financials, Healthcare, and 10-Year Treasury Yield
tickers = ['XLK', 'XLF', 'XLV', '^TNX']

In [ ]:
# 5-year period: from january 1, 2021 to October 12, 2026
start_date = '2021-01-01'
end_date = '2026-10-12'

In [ ]:
# Download historical data for the specified tickers and date range
data = yf.download(tickers, start=start_date, end=end_date)['Close']

In [ ]:
# Display the 1st 5 rows of the dataset
print(data.head(5))

In [ ]:
# Finding the number of missing values in the dataset
print("Missing values count per column:")
print(data.isnull().sum())


In [ ]:
# Calculate cumulative returns for the tickers
cumulative_returns = (data / data.iloc[0] - 1) * 100
print(cumulative_returns.tail())

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# Create a line plot for cumulative returns
plt.figure(figsize=(12, 5))
plt.plot(cumulative_returns.index, cumulative_returns['XLF'], label='XLF', linewidth=2, color='red')
plt.plot(cumulative_returns.index, cumulative_returns['XLK'], label='XLK', linewidth=2, color='green')
plt.plot(cumulative_returns.index, cumulative_returns['XLV'], label='XLV', linewidth=2, color='blue')
plt.xlabel('Year')
plt.ylabel('Cumulative Return (%)')
plt.title('Performance of Tickers')
plt.legend(cumulative_returns.columns)
plt.show()

In [ ]:
# Percentage change
analysis_df = pd.DataFrame()
analysis_df[['XLK', 'XLF', 'XLV']] = data[['XLK', 'XLF', 'XLV']].pct_change()
analysis_df['^TNX_change'] = data['^TNX'].diff()
analysis_df = analysis_df.dropna()

In [ ]:
# Correlation matrix
correlation_matrix = analysis_df.corr()
print("Correlation Matrix:")
print(correlation_matrix)

In [ ]:
pip install seaborn


In [ ]:
import seaborn as sns

In [ ]:
plt.figure(figsize=(8, 6))

In [ ]:
sns.heatmap(
    correlation_matrix, 
    annot=True,          
    cmap='coolwarm',     # Color map (blue for negative, red for positive)
    vmin=-1, vmax=1,    
    fmt='.3f',           
    linewidths=0.5
)

plt.title('Sector Returns vs. Interest Rate Changes', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np


horizons = {
    '1 Month': 21,
    '3 Months': 63,
    '6 Months': 126,
    '1 Year': 252,
    '2 Years': 504,
    '3 Years': 756
}

sectors = ['XLK', 'XLF', 'XLV']



In [ ]:
# Annualized Rolling Volatility for each horizon
volatility_summary = {}

for label, days in horizons.items():
    # Calculate daily returns, then standard deviation over rolling window, annualized
    rolling_vol = data[sectors].pct_change().rolling(window=days).std() * np.sqrt(252) * 100
    volatility_summary[label] = rolling_vol.mean()

    vol_df = pd.DataFrame(volatility_summary).T
print("--- Average Annualized Volatility (%) by Holding Period ---")
print(vol_df.round(2))

In [ ]:
def calculate_drawdown(df):
    rolling_max = df.cummax()
    drawdown = (df - rolling_max) / rolling_max
    return drawdown * 100

drawdowns = calculate_drawdown(data[sectors])

In [ ]:
plt.figure(figsize=(12, 5))

# Define a clean color dictionary for each sector
sector_colors = {
    'XLK': 'green',
    'XLF': 'red',
    'XLV': 'blue'
}

for ticker in sectors:
    plt.plot(
        drawdowns.index, 
        drawdowns[ticker], 
        label=ticker, 
        color=sector_colors.get(ticker, 'black'), 
        linewidth=1.2
    )

plt.title('Historical Drawdowns (%) — Downside Risk Over Time', fontsize=12)
plt.ylabel('Drawdown (%)')
plt.xlabel('Year')
plt.grid(True, linestyle='--', alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Ratio with 0% risk-free rate
daily_returns = data[['XLK', 'XLF', 'XLV']].pct_change().dropna()
sharpe_ratio = (daily_returns.mean() * 252) / (daily_returns.std() * np.sqrt(252))

print("--- Sharpe Ratios (Risk-Adjusted Performance) ---")
print(sharpe_ratio.round(2))

In [ ]:
def max_drawdown_duration(prices):
    # Cumulative max
    cum_max = prices.cummax()

    # Drawdown periods
    drawdown = (prices - cum_max) / cum_max
    
    # Drawdowns durations
    is_underwater = drawdown < 0
    
    durations = {}
    for col in prices.columns:
        series = is_underwater[col]
        
        group_id = (~series).cumsum()
        spell_lengths = series.groupby(group_id).sum()
        durations[col] = spell_lengths.max()
        
    return pd.Series(durations)

recovery_days = max_drawdown_duration(data[sectors])
print("--- Maximum Recovery Duration (in Days) ---")
print(recovery_days)